# LogStar — E5: 12 ca tên giống

Chạy trên Kaggle, không dùng tài nguyên inference của laptop. Hai model small/base, mỗi model một lượt, không train hoặc fine-tune.

31 tên sản phẩm làm ứng viên để giữ các giống dễ nhầm; chỉ **12 truy vấn**, trong đó 10 có nhãn, RD6 và Chuối Cavendish không chấm đúng/sai. Dữ liệu tổng hợp; chưa đại diện truy vấn người dùng. Không kết nối Supabase, không có secret.

Notebook cần **Internet** để tải model công khai. CPU Kaggle đủ cho mẫu này; GPU là tùy chọn. Các bảng cosine và top-3 xuất ra `/kaggle/working/e5-tiny-results/`.


In [ ]:
import os
assert os.path.isdir('/kaggle/working'), 'Chạy notebook này trên Kaggle; không chạy inference local.'
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'sentence-transformers==5.7.0', 'transformers==5.19.0'])
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


In [ ]:
import gc, json, time, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from IPython.display import display

torch.set_num_threads(2)
torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Kaggle device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
OUT = Path('/kaggle/working/e5-tiny-results')
OUT.mkdir(exist_ok=True)


In [ ]:
PAYLOAD = json.loads('{"catalog": [{"id": "durian-0", "group": "durian", "fruit": "Sầu riêng", "variety": "Ri6", "name": "Sầu riêng Ri6", "aliases": ["Ri 6", "RI-6"]}, {"id": "durian-1", "group": "durian", "fruit": "Sầu riêng", "variety": "Monthong", "name": "Sầu riêng Monthong", "aliases": ["Dona", "Đô Na", "Mon Thong"]}, {"id": "durian-2", "group": "durian", "fruit": "Sầu riêng", "variety": "Musang King", "name": "Sầu riêng Musang King", "aliases": ["Musangking"]}, {"id": "durian-3", "group": "durian", "fruit": "Sầu riêng", "variety": "Black Thorn", "name": "Sầu riêng Black Thorn", "aliases": ["Blackthorn"]}, {"id": "durian-4", "group": "durian", "fruit": "Sầu riêng", "variety": "Chín Hóa", "name": "Sầu riêng Chín Hóa", "aliases": []}, {"id": "durian-5", "group": "durian", "fruit": "Sầu riêng", "variety": "Khổ qua xanh", "name": "Sầu riêng Khổ qua xanh", "aliases": []}, {"id": "durian-6", "group": "durian", "fruit": "Sầu riêng", "variety": "Chuồng bò", "name": "Sầu riêng Chuồng bò", "aliases": []}, {"id": "mango-0", "group": "mango", "fruit": "Xoài", "variety": "Cát Hòa Lộc", "name": "Xoài Cát Hòa Lộc", "aliases": []}, {"id": "mango-1", "group": "mango", "fruit": "Xoài", "variety": "Cát Chu", "name": "Xoài Cát Chu", "aliases": []}, {"id": "mango-2", "group": "mango", "fruit": "Xoài", "variety": "Keo", "name": "Xoài Keo", "aliases": []}, {"id": "mango-3", "group": "mango", "fruit": "Xoài", "variety": "Tượng da xanh", "name": "Xoài Tượng da xanh", "aliases": []}, {"id": "pomelo-0", "group": "pomelo", "fruit": "Bưởi", "variety": "Da xanh", "name": "Bưởi Da xanh", "aliases": []}, {"id": "pomelo-1", "group": "pomelo", "fruit": "Bưởi", "variety": "Năm Roi", "name": "Bưởi Năm Roi", "aliases": []}, {"id": "pomelo-2", "group": "pomelo", "fruit": "Bưởi", "variety": "Diễn", "name": "Bưởi Diễn", "aliases": []}, {"id": "pomelo-3", "group": "pomelo", "fruit": "Bưởi", "variety": "Đoan Hùng", "name": "Bưởi Đoan Hùng", "aliases": []}, {"id": "jackfruit-0", "group": "jackfruit", "fruit": "Mít", "variety": "Thái", "name": "Mít Thái", "aliases": []}, {"id": "jackfruit-1", "group": "jackfruit", "fruit": "Mít", "variety": "Ruột đỏ", "name": "Mít Ruột đỏ", "aliases": []}, {"id": "jackfruit-2", "group": "jackfruit", "fruit": "Mít", "variety": "Tố nữ", "name": "Mít Tố nữ", "aliases": []}, {"id": "longan-0", "group": "longan", "fruit": "Nhãn", "variety": "Xuồng cơm vàng", "name": "Nhãn Xuồng cơm vàng", "aliases": []}, {"id": "longan-1", "group": "longan", "fruit": "Nhãn", "variety": "Tiêu da bò", "name": "Nhãn Tiêu da bò", "aliases": []}, {"id": "longan-2", "group": "longan", "fruit": "Nhãn", "variety": "Ido", "name": "Nhãn Ido", "aliases": []}, {"id": "dragonfruit-0", "group": "dragonfruit", "fruit": "Thanh long", "variety": "Ruột trắng", "name": "Thanh long Ruột trắng", "aliases": []}, {"id": "dragonfruit-1", "group": "dragonfruit", "fruit": "Thanh long", "variety": "Ruột đỏ LĐ1", "name": "Thanh long Ruột đỏ LĐ1", "aliases": ["Long Định 1", "LĐ1", "LD1"]}, {"id": "dragonfruit-2", "group": "dragonfruit", "fruit": "Thanh long", "variety": "Ruột tím hồng LĐ5", "name": "Thanh long Ruột tím hồng LĐ5", "aliases": ["Long Định 5", "LĐ5", "LD5"]}, {"id": "rice-0", "group": "rice", "fruit": "Lúa", "variety": "ST24", "name": "Lúa ST24", "aliases": []}, {"id": "rice-1", "group": "rice", "fruit": "Lúa", "variety": "ST25", "name": "Lúa ST25", "aliases": []}, {"id": "rice-2", "group": "rice", "fruit": "Lúa", "variety": "OM5451", "name": "Lúa OM5451", "aliases": []}, {"id": "rice-3", "group": "rice", "fruit": "Lúa", "variety": "OM18", "name": "Lúa OM18", "aliases": []}, {"id": "rice-4", "group": "rice", "fruit": "Lúa", "variety": "IR50404", "name": "Lúa IR50404", "aliases": []}, {"id": "grape-0", "group": "grape", "fruit": "Nho", "variety": "NH01-48", "name": "Nho NH01-48", "aliases": []}, {"id": "grape-1", "group": "grape", "fruit": "Nho", "variety": "NH01-152", "name": "Nho NH01-152", "aliases": []}], "queries": [{"id": "q001", "text": "Sầu riêng Ri6", "target": "durian-0", "category": "exact", "note": ""}, {"id": "q156", "text": "Sầu riêng RD6", "target": null, "category": "ambiguous", "note": "Không có một đáp án duy nhất; chỉ quan sát top/margin, không tính accuracy."}, {"id": "q140", "text": "Sầu riêng Dona", "target": "durian-1", "category": "alias", "note": "Alias trong fixture; thí nghiệm enriched cung cấp alias này cho passage một cách công khai."}, {"id": "q002", "text": "sau rieng ri6", "target": "durian-0", "category": "unaccented", "note": ""}, {"id": "q009", "text": "Sầu riêng Musang King", "target": "durian-2", "category": "exact", "note": ""}, {"id": "q148", "text": "Cần Ri6, không mua Monthong", "target": "durian-0", "category": "hard_context", "note": ""}, {"id": "q033", "text": "Xoài Cát Chu", "target": "mango-1", "category": "exact", "note": ""}, {"id": "q152", "text": "Xoài Cát Chu, không lấy Cát Hòa Lộc", "target": "mango-1", "category": "hard_context", "note": ""}, {"id": "q049", "text": "Bưởi Năm Roi", "target": "pomelo-1", "category": "exact", "note": ""}, {"id": "q101", "text": "Lúa ST25", "target": "rice-1", "category": "exact", "note": ""}, {"id": "q154", "text": "Gạo ST25, không phải ST24", "target": "rice-1", "category": "hard_context", "note": ""}, {"id": "q164", "text": "Chuối Cavendish", "target": null, "category": "out_of_catalog", "note": "Không có trong catalog đóng; cosine top cao vẫn không chứng minh có giống phù hợp."}], "version": "kaggle-tiny-12-v1"}')

catalog, queries = PAYLOAD['catalog'], PAYLOAD['queries']
names = [p['name'] for p in catalog]
print(len(catalog), 'ứng viên;', len(queries), 'truy vấn; ', sum(q['target'] is not None for q in queries), 'ca có nhãn')
display(pd.DataFrame(queries)[['text','target','category']])


In [ ]:
MODEL_SPECS = [
    ('small', 'intfloat/multilingual-e5-small', '614241f622f53c4eeff9890bdc4f31cfecc418b3'),
    ('base', 'intfloat/multilingual-e5-base', 'd13f1b27baf31030b7fd040960d60d909913633f'),
]
all_rows, summaries, score_rows = [], [], []
for label, model_id, revision in MODEL_SPECS:
    print('Loading', model_id, flush=True)
    started = time.perf_counter()
    model = SentenceTransformer(model_id, revision=revision, device=DEVICE, trust_remote_code=False)
    model.max_seq_length = 512
    loaded = time.perf_counter()-started
    if DEVICE == 'cuda': torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        passages = model.encode(['passage: '+n for n in names], batch_size=8,
                                normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        vectors = model.encode(['query: '+q['text'] for q in queries], batch_size=8,
                               normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    encode_seconds = time.perf_counter()-started
    cosine = vectors @ passages.T
    assert np.isfinite(cosine).all()
    hit1, hit3, reciprocal, count = 0, 0, 0., 0
    for q, scores in zip(queries, cosine):
        order = sorted(range(len(catalog)), key=lambda i: (-float(scores[i]), catalog[i]['id']))
        target_index = next((i for i,p in enumerate(catalog) if p['id']==q['target']), None)
        rank = order.index(target_index)+1 if target_index is not None else None
        if rank is not None:
            count+=1; hit1+=int(rank==1); hit3+=int(rank<=3); reciprocal+=1/rank
        top = order[:3]
        all_rows.append({'model': label, 'query': q['text'], 'category': q['category'],
                         'expected': names[target_index] if target_index is not None else 'Không gán đích',
                         'target_rank': rank, 'top1': names[top[0]], 'cosine_top1': float(scores[top[0]]),
                         'top2': names[top[1]], 'cosine_top2': float(scores[top[1]]),
                         'top3': names[top[2]], 'cosine_top3': float(scores[top[2]]),
                         'top1_top2_margin': float(scores[top[0]]-scores[top[1]])})
        for position, i in enumerate(order, 1):
            score_rows.append({'model': label, 'query': q['text'], 'target': q['target'],
                               'rank': position, 'candidate': names[i], 'cosine': float(scores[i])})
    summaries.append({'model': label, 'model_id': model_id, 'revision': revision,
                      'labelled_queries': count, 'top1_correct': hit1, 'top1_accuracy': hit1/count,
                      'hit_at_3': hit3/count, 'mrr': reciprocal/count,
                      'device': DEVICE, 'model_load_seconds': loaded, 'encode_all_seconds': encode_seconds})
    del model, passages, vectors, cosine
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(label, 'finished', flush=True)


In [ ]:
summary = pd.DataFrame(summaries)
details = pd.DataFrame(all_rows)
display(summary[['model','top1_correct','labelled_queries','top1_accuracy','hit_at_3','mrr','device','encode_all_seconds']])
display(details[['model','query','expected','top1','cosine_top1','top2','cosine_top2','target_rank']])
summary.to_csv(OUT/'summary.csv', index=False)
details.to_csv(OUT/'top3.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(score_rows).to_csv(OUT/'cosine-all.csv', index=False, encoding='utf-8-sig')
result = {'created_at': datetime.now(timezone.utc).isoformat(), 'environment': {
    'python': platform.python_version(), 'torch': torch.__version__,
    'sentence_transformers': importlib.metadata.version('sentence-transformers'),
    'transformers': importlib.metadata.version('transformers'), 'device': DEVICE},
    'fixture': PAYLOAD, 'summary': summaries, 'rows': all_rows,
    'limits': '12 author-labelled probes only, 10 scored; not comparable to the larger local probe. No cosine threshold is calibrated. RD6 is unlabelled.'}
(OUT/'results.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('DONE — kết quả:', str(OUT))


## Đọc kết quả

- So điểm đúng giống và giống sai gần nhất; cosine không phải xác suất đúng.
- Với RD6: chỉ xem các gợi ý, chưa coi RD6 là alias đã xác nhận của Ri6.
- Câu có `không lấy/không mua` cần kiểm tra riêng: embedding có thể vẫn ưu tiên giống bị phủ định.
- Mẫu nhỏ dùng để kiểm tra nhanh; không chọn ngưỡng production từ 10 ca có nhãn.
